# Dynamic Programming for Reinforcement Learning

## Model-Based vs Model-Free

| | Model-Based | Model-Free |
|--|-------------|------------|
| Knows $P(s'|s,a)$ | ✅ Yes | ❌ No |
| Knows $R(s,a)$ | ✅ Yes | ❌ No |
| Examples | Dynamic Programming | Q-Learning, Monte Carlo |
| Data efficiency | High | Low |
| Applicability | Narrow | Broad |

**Dynamic Programming (DP)** assumes we have a **perfect model** of the environment. It uses the Bellman equations as update rules.

---

## Policy Evaluation (Prediction)

**Goal**: Given policy $\pi$, compute $V^\pi(s)$ for all states.

**Iterative Policy Evaluation**: Apply Bellman expectation as an update rule repeatedly:

$$V_{k+1}(s) = \sum_a \pi(a|s) \sum_{s',r} p(s',r|s,a)\left[r + \gamma V_k(s')\right]$$

This converges to $V^\pi$ as $k \to \infty$.

---

## Policy Improvement

**Policy Improvement Theorem**: If $Q^\pi(s, \pi'(s)) \geq V^\pi(s)$ for all $s$, then $V^{\pi'}(s) \geq V^\pi(s)$ for all $s$.

**Greedy policy improvement**:

$$\pi'(s) = \arg\max_a Q^\pi(s,a) = \arg\max_a \sum_{s',r} p(s',r|s,a)\left[r + \gamma V^\pi(s')\right]$$

---

## Policy Iteration

Alternate between evaluation and improvement:

$$\pi_0 \xrightarrow{E} V^{\pi_0} \xrightarrow{I} \pi_1 \xrightarrow{E} V^{\pi_1} \xrightarrow{I} \pi_2 \cdots \xrightarrow{} \pi^*$$

**Algorithm**:
1. Initialize $V(s)$ arbitrarily, $\pi$ arbitrarily
2. **Policy Evaluation**: Update $V$ until convergence under $\pi$
3. **Policy Improvement**: Update $\pi$ greedily w.r.t. $V$
4. If policy changed, go to step 2. Otherwise done.

---

## Value Iteration

**Key insight**: Combine evaluation and improvement in one step using the Bellman **optimality** equation:

$$V_{k+1}(s) = \max_a \sum_{s',r} p(s',r|s,a)\left[r + \gamma V_k(s')\right]$$

Only one sweep per iteration (no inner loop). More efficient than policy iteration in practice.

**Convergence**: $\|V_{k+1} - V_k\|_\infty < \theta$ (small threshold)

---

## Generalized Policy Iteration (GPI)

Both policy iteration and value iteration are special cases of **GPI** any interleaving of evaluation and improvement steps:

$$V \leftrightarrow \pi$$

The value function and policy pull toward each other, converging to optimal.

---

## Asynchronous DP

Instead of sweeping all states, update states **one at a time** in any order. Useful when:
- State space is very large
- Some states more important than others
- Real-time applications

**Prioritized sweeping**: Update states with the largest Bellman error first.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# ─── GridWorld Environment ───
class GridWorld:
    """
    4x4 GridWorld:
    S . . .
    . X . .
    . . . .
    . . . G
    S=start(0,0), G=goal(3,3), X=wall(1,1)
    """
    def __init__(self, size=4, gamma=0.9):
        self.size = size
        self.gamma = gamma
        self.n_states = size * size
        self.n_actions = 4  # 0=up, 1=right, 2=down, 3=left
        self.goal = (3, 3)
        self.wall = (1, 1)
        self.actions = [(-1,0),(0,1),(1,0),(0,-1)]  # up,right,down,left

    def state_to_idx(self, r, c): return r * self.size + c
    def idx_to_state(self, idx): return idx // self.size, idx % self.size

    def step(self, state_idx, action):
        r, c = self.idx_to_state(state_idx)
        if (r, c) == self.goal:
            return state_idx, 0, True
        dr, dc = self.actions[action]
        nr, nc = r + dr, c + dc
        # Stay if out of bounds or wall
        if 0 <= nr < self.size and 0 <= nc < self.size and (nr,nc) != self.wall:
            r, c = nr, nc
        reward = 10 if (r,c) == self.goal else -1
        done = (r,c) == self.goal
        return self.state_to_idx(r, c), reward, done

env = GridWorld()
print(f"States: {env.n_states}, Actions: {env.n_actions}")

States: 16, Actions: 4


In [2]:
# ─── Policy Evaluation ───
def policy_evaluation(env, policy, theta=1e-6):
    """Iteratively evaluate a given policy."""
    V = np.zeros(env.n_states)
    iterations = 0
    while True:
        delta = 0
        for s in range(env.n_states):
            r, c = env.idx_to_state(s)
            if (r,c) == env.goal or (r,c) == env.wall:
                continue
            v = V[s]
            # V(s) = sum_a pi(a|s) * sum_{s',r} p(s',r|s,a)[r + gamma*V(s')]
            V[s] = sum(
                policy[s, a] * (rew + env.gamma * V[ns])
                for a in range(env.n_actions)
                for ns, rew, _ in [env.step(s, a)]
            )
            delta = max(delta, abs(v - V[s]))
        iterations += 1
        if delta < theta:
            break
    return V, iterations

# Uniform random policy
random_policy = np.ones((env.n_states, env.n_actions)) / env.n_actions
V_random, iters = policy_evaluation(env, random_policy)
print(f"Policy evaluation converged in {iters} iterations")
print("V(s) under random policy:")
print(V_random.reshape(4, 4).round(2))

Policy evaluation converged in 81 iterations
V(s) under random policy:
[[-8.81 -8.54 -7.63 -7.09]
 [-8.54  0.   -6.19 -5.27]
 [-7.63 -6.19 -3.98 -0.43]
 [-7.09 -5.27 -0.43  0.  ]]


In [3]:
# ─── Policy Iteration ───
def policy_iteration(env, theta=1e-6):
    policy = np.ones((env.n_states, env.n_actions)) / env.n_actions
    pi_iters = 0

    while True:
        # Evaluation
        V, _ = policy_evaluation(env, policy, theta)

        # Improvement
        policy_stable = True
        for s in range(env.n_states):
            r, c = env.idx_to_state(s)
            if (r,c) == env.goal or (r,c) == env.wall:
                continue
            old_action = np.argmax(policy[s])
            # Q(s,a) = sum_{s',r} p(s',r|s,a)[r + gamma*V(s')]
            q_vals = [env.step(s, a)[1] + env.gamma * V[env.step(s, a)[0]]
                      for a in range(env.n_actions)]
            best_action = np.argmax(q_vals)
            policy[s] = 0
            policy[s, best_action] = 1.0
            if old_action != best_action:
                policy_stable = False
        pi_iters += 1
        if policy_stable:
            break

    return policy, V, pi_iters

opt_policy, V_opt, pi_iters = policy_iteration(env)
print(f"Policy iteration converged in {pi_iters} iterations")
print("Optimal V(s):")
print(V_opt.reshape(4,4).round(2))

action_symbols = ['↑','→','↓','←']
policy_grid = [[action_symbols[np.argmax(opt_policy[env.state_to_idx(r,c)])]
                if (r,c) != env.goal and (r,c) != env.wall else ('G' if (r,c)==env.goal else 'X')
                for c in range(4)] for r in range(4)]
print("\nOptimal Policy:")
for row in policy_grid:
    print(' '.join(row))

Policy iteration converged in 3 iterations
Optimal V(s):
[[ 1.81  3.12  4.58  6.2 ]
 [ 3.12  0.    6.2   8.  ]
 [ 4.58  6.2   8.   10.  ]
 [ 6.2   8.   10.    0.  ]]

Optimal Policy:
→ → → ↓
↓ X → ↓
→ → → ↓
→ → → G


In [4]:
# ─── Value Iteration ───
def value_iteration(env, theta=1e-6):
    V = np.zeros(env.n_states)
    vi_iters = 0

    while True:
        delta = 0
        for s in range(env.n_states):
            r, c = env.idx_to_state(s)
            if (r,c) == env.goal or (r,c) == env.wall:
                continue
            v = V[s]
            # V(s) = max_a [r + gamma*V(s')]
            V[s] = max(
                env.step(s, a)[1] + env.gamma * V[env.step(s, a)[0]]
                for a in range(env.n_actions)
            )
            delta = max(delta, abs(v - V[s]))
        vi_iters += 1
        if delta < theta:
            break

    # Extract policy
    policy = np.zeros((env.n_states, env.n_actions))
    for s in range(env.n_states):
        r, c = env.idx_to_state(s)
        if (r,c) == env.goal or (r,c) == env.wall:
            continue
        q_vals = [env.step(s,a)[1] + env.gamma*V[env.step(s,a)[0]] for a in range(env.n_actions)]
        policy[s, np.argmax(q_vals)] = 1.0

    return policy, V, vi_iters

vi_policy, V_vi, vi_iters = value_iteration(env)
print(f"Value iteration converged in {vi_iters} iterations")
print("Value function (Value Iteration):")
print(V_vi.reshape(4,4).round(2))
print("\nValues match Policy Iteration:", np.allclose(V_opt, V_vi, atol=1e-3))

Value iteration converged in 7 iterations
Value function (Value Iteration):
[[ 1.81  3.12  4.58  6.2 ]
 [ 3.12  0.    6.2   8.  ]
 [ 4.58  6.2   8.   10.  ]
 [ 6.2   8.   10.    0.  ]]

Values match Policy Iteration: True


## Additional Learning Resources

### Books
- **Sutton & Barto Ch. 4** Dynamic Programming: http://incompleteideas.net/book/the-book-2nd.html
- **Algorithms for Reinforcement Learning** Szepesvári (free PDF): https://sites.ualberta.ca/~szepesva/papers/RLAlgsInMDPs.pdf

### Courses
- **David Silver Lecture 3** Planning by DP: https://www.davidsilver.uk/teaching/
- **Spinning Up** OpenAI: https://spinningup.openai.com/

### Libraries
- **Gymnasium** (GridWorld and more): https://gymnasium.farama.org/
- **MDPtoolbox** (Python): https://pymdptoolbox.readthedocs.io/